# 08 — V3 Single-Panel Egitim (In-Silico Skorlar ILE)

**Amac**: In-silico skorlar ve label-encoded prediction'lar KORUNARAK
single-panel modeller egitilir. 5 model tipi (LightGBM, XGBoost, NN, DNN, SVM),
hold-out degerlendirme, CFTR testi ve cross-model hata analizi.

**Paneller**: General, Hereditary_Cancer, PAH (CFTR → General modeli ile test)

In [8]:
# Cell 1: Imports & Config
import sys, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, DATA_PATH, PANELS_SINGLE, RESULTS_WITH_INSIL_DIR, REPORTS_DIR
from src.features import prepare_data_v3
from src.models import (grid_search_lightgbm, grid_search_xgboost,
                        grid_search_nn_fast, grid_search_dnn_fast, grid_search_svm)
from src.metrics import optimize_threshold, compute_all_metrics
from src.utils import prepare_for_xgb, prepare_for_nn

os.makedirs(RESULTS_WITH_INSIL_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

MODEL_FUNCS = {
    'lightgbm': grid_search_lightgbm,
    'xgboost':  grid_search_xgboost,
    'nn':       grid_search_nn_fast,
    'dnn':      grid_search_dnn_fast,
    'svm':      grid_search_svm,
}

RESULTS_DIR = RESULTS_WITH_INSIL_DIR
print(f'Proje koku: {PROJECT_ROOT}')
print(f'Sonuc dizini: {RESULTS_DIR}')

Proje koku: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model
Sonuc dizini: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\v3_with_insil_panels


In [9]:
# Cell 2: Data Loading & Feature Engineering (IN-SILICO ILE)
df_raw = pd.read_csv(DATA_PATH)
print(f'Ham veri: {df_raw.shape}')

# Target derivation
sig = df_raw['clinvar__sig'].str.lower().str.strip()
target_map = {
    'benign': 0, 'likely benign': 0,
    'pathogenic': 1, 'likely pathogenic': 1,
}
df_raw['target'] = sig.map(target_map)
df_raw = df_raw.dropna(subset=['target'])
df_raw['target'] = df_raw['target'].astype(int)
print(f'Target dagilimi: {df_raw["target"].value_counts()}')

# Panel bilgisini sakla
panel_series = df_raw['Panel'].copy()

# Feature engineering (in-silico ILE - v3 pipeline)
df_v3 = prepare_data_v3(df_raw)

# Panel i geri ekle
df_v3['Panel'] = panel_series.values

print(f'FE sonrasi: {df_v3.shape}')
print(f'Panel dagilimi: {df_v3["Panel"].value_counts()}')

# Panel basina sinif dagilimi kontrol et
# min_class < 5 olan paneller egitilemez -> test-only
TRAIN_PANELS = []
TEST_ONLY_PANELS = []
for p in PANELS_SINGLE:
    p_df = df_v3[df_v3['Panel'] == p]
    p_target = p_df['target']
    vc = p_target.value_counts()
    min_class = vc.min()
    print(f'  {p}: {len(p_df)} satir, pos={vc.get(1,0)}, neg={vc.get(0,0)}, min_class={min_class}')
    if min_class < 5:
        TEST_ONLY_PANELS.append(p)
        print(f'    -> test-only (min_class < 5)')
    else:
        TRAIN_PANELS.append(p)

# CFTR ve test-only panelleri ayir
df_cftr = df_v3[df_v3['Panel'] == 'CFTR'].copy()
df_test_only = {p: df_v3[df_v3['Panel'] == p].copy() for p in TEST_ONLY_PANELS}
df_panels = df_v3[df_v3['Panel'].isin(TRAIN_PANELS)].copy()

print(f'Egitim panelleri: {TRAIN_PANELS}')
print(f'Test-only paneller: {TEST_ONLY_PANELS + ["CFTR"]}')
print(f'CFTR: {len(df_cftr)} satir')
print(f'Egitim havuzu: {len(df_panels)} satir')

Ham veri: (4287, 119)
Target dagilimi: target
1    2951
0    1336
Name: count, dtype: int64
  LabelEncode esm1b__prediction: 2 sinif
  LabelEncode metalr__pred: 2 sinif
  LabelEncode metarnn__pred: 2 sinif
  LabelEncode metasvm__pred: 2 sinif
  LabelEncode mistic__pred: 2 sinif
  LabelEncode mutationtaster__prediction: 4 sinif
  LabelEncode phdsnpg__prediction: 2 sinif
  LabelEncode provean__prediction: 2 sinif
  LabelEncode sift__prediction: 2 sinif
  LabelEncode alphamissense__am_class: 3 sinif
  OHE (ayri alfabe): 48 feature
  K-mer DNA_11mer_Ref: 16 2-mer feature
  K-mer DNA_11mer_Alt: 16 2-mer feature
  K-mer Prot_11mer_Ref: 430 2-mer feature
  K-mer Prot_11mer_Alt: 434 2-mer feature
  57 gereksiz sutun kaldirildi
  r>0.99 kopya filtreleme: 15 sutun dusuruldu
[V3] Veri boyutu: 4287 x 991
FE sonrasi: (4287, 991)
Panel dagilimi: Panel
General              3156
Hereditary_Cancer     715
PAH                   324
CFTR                   92
Name: count, dtype: int64
  General: 3156 sati

In [10]:
# Cell 3: Per-Panel Egitim (Hold-Out + Grid Search)
all_results = []
all_predictions = {}   # {panel: {'y_true': Series, 'preds': {model: array}, 'indices': Index}}
trained_models = {}    # {(panel, model_name): (model_obj, threshold, extras)}
cftr_results = []      # CFTR tahminleri

# Orijinal veri bilgilerini error analysis icin sakla
original_info_cols = ['base__chrom', 'base__pos', 'base__ref_base', 'base__alt_base', 'clinvar__sig']
df_original_info = df_raw[original_info_cols].copy()

for panel in TRAIN_PANELS:
    print(f"\n{'='*70}")
    print(f"PANEL: {panel}")
    print(f"{'='*70}")

    df_panel = df_panels[df_panels['Panel'] == panel].copy()
    df_panel.drop(columns=['Panel'], inplace=True)

    X = df_panel.drop(columns=['target'])
    y = df_panel['target']
    print(f"  Toplam: {len(y)} satir (pos={y.sum()}, neg={(y==0).sum()})")

    # Hold-out split: %80 CV, %20 hold-out
    X_cv, X_holdout, y_cv, y_holdout = train_test_split(
        X, y, test_size=0.20, random_state=SEED, stratify=y
    )
    print(f"  CV seti: {len(y_cv)} | Hold-out: {len(y_holdout)}")

    panel_preds = {}

    for mt_name, builder in MODEL_FUNCS.items():
        print(f"\n  --- {mt_name.upper()} ---")
        t0 = time.time()

        # Grid search: CV setinden train/val split dahili yapilir
        cat_features = X_cv.select_dtypes(include=['category']).columns.tolist()

        # SVM 5 deger dondurur (scaler dahil)
        result = builder(X_cv, y_cv, X_holdout, y_holdout, cat_features)
        if mt_name == 'svm':
            model, best_combo, best_thr, y_prob_ho, svm_scaler = result
        else:
            model, best_combo, best_thr, y_prob_ho = result
            svm_scaler = None

        elapsed = time.time() - t0

        # Hold-out degerlendirme
        y_pred_ho = (y_prob_ho >= best_thr).astype(int)
        metrics = compute_all_metrics(y_holdout, y_pred_ho, y_prob_ho)

        # Error analysis icin tahminleri kaydet
        panel_preds[mt_name] = pd.Series(y_pred_ho.ravel(), index=X_holdout.index)

        # Modeli sakla
        trained_models[(panel, mt_name)] = (model, best_thr, svm_scaler, cat_features)

        result_row = {
            'panel': panel, 'model': mt_name, 'fe_mode': 'v3_with_insil',
            'threshold': best_thr, **metrics,
            'elapsed_sec': round(elapsed, 1),
            'best_params': str(best_combo),
            'n_train': len(y_cv), 'n_test': len(y_holdout),
        }
        all_results.append(result_row)
        print(f"  F1={metrics['f1']:.4f}  AUC={metrics['auc_roc']:.4f}  "
              f"P={metrics['precision']:.4f}  R={metrics['recall']:.4f}  ({elapsed:.1f}s)")

        # ---- CFTR + test-only panel testi (sadece General paneli icin) ----
        if panel == 'General':
            test_panels_data = {'CFTR': df_cftr}
            test_panels_data.update(df_test_only)

            for tp_name, tp_df in test_panels_data.items():
                if len(tp_df) == 0:
                    continue
                X_tp = tp_df.drop(columns=['target', 'Panel'])
                y_tp = tp_df['target']

                try:
                    if mt_name == 'svm':
                        X_tp_enc = X_tp.copy()
                        if cat_features:
                            X_tp_enc, _, _ = prepare_for_xgb(X_tp_enc, X_tp_enc, cat_features)
                        X_tp_enc = X_tp_enc.fillna(0).astype(float)
                        X_tp_scaled = svm_scaler.transform(X_tp_enc)
                        y_prob_tp = model.predict_proba(X_tp_scaled)[:, 1]
                    elif mt_name == 'xgboost':
                        X_tp_enc = X_tp.copy()
                        for col in X_tp_enc.select_dtypes(include=['category']).columns:
                            X_tp_enc[col] = X_tp_enc[col].cat.codes
                        y_prob_tp = model.predict_proba(X_tp_enc)[:, 1]
                    elif mt_name in ('nn', 'dnn'):
                        import torch
                        nn_data = prepare_for_nn(X_cv, X_tp, y_cv, y_tp, cat_features)
                        _, X_tp_t, _, _, _, _, _ = nn_data
                        model.eval()
                        with torch.no_grad():
                            y_prob_tp = torch.sigmoid(model(X_tp_t)).numpy()
                    else:  # lightgbm
                        y_prob_tp = model.predict_proba(X_tp)[:, 1]

                    y_pred_tp = (y_prob_tp >= best_thr).astype(int)
                    tp_metrics = compute_all_metrics(y_tp, y_pred_tp, y_prob_tp)

                    cftr_results.append({
                        'test_panel': tp_name, 'model': mt_name,
                        'threshold': best_thr,
                        'n_test': len(y_tp),
                        **tp_metrics,
                    })
                    print(f"    {tp_name} Test: F1={tp_metrics['f1']:.4f} "
                          f"P={tp_metrics['precision']:.4f} R={tp_metrics['recall']:.4f} "
                          f"Acc={tp_metrics['balanced_accuracy']:.4f} ({len(y_tp)} satir)")
                except Exception as e:
                    print(f"    {tp_name} testi basarisiz: {e}")

        all_predictions[panel] = {
        'y_true': y_holdout,
        'preds': panel_preds,
        'indices': X_holdout.index,
    }

# Sonuclari kaydet
results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(RESULTS_DIR, 'panel_results.csv'), index=False)
print(f"\n\nSonuclar kaydedildi: {RESULTS_DIR}/panel_results.csv")

if cftr_results:
    cftr_df = pd.DataFrame(cftr_results)
    cftr_df.to_csv(os.path.join(RESULTS_DIR, 'cftr_test_results.csv'), index=False)

print('\n=== SONUC TABLOSU ===')
display(results_df[['panel','model','f1','auc_roc','precision','recall','mcc','elapsed_sec']])


PANEL: General
  Toplam: 3156 satir (pos=2182.0, neg=974)
  CV seti: 2524 | Hold-out: 632

  --- LIGHTGBM ---
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 100, 'num_leaves': 63, 'learning_rate': 0.05} -> CV F1=0.9842
  F1=0.9784  AUC=0.9960  P=0.9729  R=0.9840  (30.7s)
    CFTR testi basarisiz: not enough values to unpack (expected 4, got 1)
    PAH Test: F1=0.9969 P=0.9969 R=0.9969 Acc=0.4985 (324 satir)

  --- XGBOOST ---
  XGBoost Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1} -> CV F1=0.9842
  F1=0.9782  AUC=0.9956  P=0.9816  R=0.9748  (40.6s)
    CFTR testi basarisiz: not enough values to unpack (expected 4, got 1)
    PAH Test: F1=0.9984 P=1.0000 R=0.9969 Acc=0.9985 (324 satir)

  --- NN ---
  NN-Fast Grid Search: 4 kombinasyon
  En iyi combo: {'dropout': 0.2, 'lr': 0.001} -> CV F1=0.9573
  F1=0.9612  AUC=0.9728  P=0.9590  R=0.9634  (209.5s)
    CFTR Test: F1=0.9945 P=1.0000 R=0.9891 Acc=0.9891 (

,panel,model,f1,auc_roc,precision,recall,mcc,elapsed_sec
0,General,lightgbm,0.978385,0.996022,0.972851,0.983982,0.929201,30.7
1,General,xgboost,0.978186,0.995646,0.981567,0.974828,0.929899,40.6
2,General,nn,0.961187,0.972798,0.958998,0.963387,0.873584,209.5
3,General,dnn,0.953356,0.978613,0.947964,0.958810,0.847022,43.4
4,General,svm,0.966781,0.982926,0.967890,0.965675,0.892619,142.4
5,Hereditary_Cancer,lightgbm,0.972603,0.992371,0.946667,1.000000,0.945555,10.3
6,Hereditary_Cancer,xgboost,0.965986,0.991882,0.934211,1.000000,0.932381,29.9
7,Hereditary_Cancer,nn,0.925170,0.975743,0.894737,0.957746,0.848296,23.6
8,Hereditary_Cancer,dnn,0.927152,0.971440,0.875000,0.985915,0.853055,7.2
9,Hereditary_Cancer,svm,0.929577,0.977308,0.929577,0.929577,0.860133,7.2


In [11]:
# Cell 4: Test-Only Panel Sonuclari (CFTR + PAH)
if cftr_results:
    print('\n=== TEST-ONLY PANEL SONUCLARI (General modeli ile) ===')
    cftr_df = pd.DataFrame(cftr_results)
    display(cftr_df[['test_panel','model','f1','precision','recall','auc_roc','mcc','balanced_accuracy','n_test']])
else:
    print('Test-only panel sonucu yok.')


=== TEST-ONLY PANEL SONUCLARI (General modeli ile) ===


,test_panel,model,f1,precision,recall,auc_roc,mcc,balanced_accuracy,n_test
0,PAH,lightgbm,0.996904,0.996904,0.996904,0.996904,-0.003096,0.498452,324
1,PAH,xgboost,0.998450,1.000000,0.996904,0.996904,0.706011,0.998452,324
2,CFTR,nn,0.994536,1.000000,0.989130,0.000000,0.000000,0.989130,92
3,PAH,nn,0.977848,1.000000,0.956656,0.981424,0.252541,0.978328,324
4,CFTR,dnn,0.994536,1.000000,0.989130,0.000000,0.000000,0.989130,92
5,PAH,dnn,0.974603,1.000000,0.950464,0.993808,0.236452,0.975232,324
6,CFTR,svm,0.994536,1.000000,0.989130,0.000000,0.000000,0.989130,92
7,PAH,svm,0.984277,1.000000,0.969040,0.990712,0.296807,0.984520,324


In [12]:
# Cell 5: Cross-Model Error Analizi
error_summaries = []

for panel in TRAIN_PANELS:
    print(f"\n{'='*50}")
    print(f"ERROR ANALIZI: {panel}")
    print(f"{'='*50}")

    data = all_predictions[panel]
    y_true = data['y_true']
    preds = data['preds']
    indices = data['indices']

    # Error matrix olustur
    error_df = pd.DataFrame(index=indices)
    error_df['y_true'] = y_true.values

    for mt_name, pred_series in preds.items():
        error_df[f'{mt_name}_pred'] = pred_series.values
        error_df[f'{mt_name}_error'] = (pred_series.values != y_true.values).astype(int)

    # Difficulty score = yanlis tahmin eden model sayisi
    error_cols = [c for c in error_df.columns if c.endswith('_error')]
    error_df['difficulty_score'] = error_df[error_cols].sum(axis=1)

    # Orijinal varyant bilgilerini ekle
    for col in original_info_cols:
        if col in df_original_info.columns:
            error_df[col] = df_original_info.loc[indices, col].values

    # Kaydet
    error_df.to_csv(os.path.join(RESULTS_DIR, f'{panel}_error_analysis.csv'))

    n_models = len(preds)
    unanimous_errors = error_df[error_df['difficulty_score'] == n_models]
    majority_errors = error_df[error_df['difficulty_score'] >= (n_models // 2 + 1)]

    print(f"  Hold-out test satirlari: {len(error_df)}")
    print(f"  Tum modeller yanlis (score={n_models}): {len(unanimous_errors)} satir")
    print(f"  Cogunluk yanlis (score>={n_models//2+1}): {len(majority_errors)} satir")
    print(f"\n  Difficulty score dagilimi:")
    print(f"  {error_df['difficulty_score'].value_counts().sort_index().to_dict()}")

    if len(unanimous_errors) > 0:
        print(f"\n  --- TUM MODELLER YANLIS ({len(unanimous_errors)} satir) ---")
        show_cols = ['y_true', 'difficulty_score'] + [c for c in original_info_cols if c in error_df.columns]
        display(unanimous_errors[show_cols].head(20))

    error_summaries.append({
        'panel': panel, 'n_test': len(error_df),
        'all_wrong': len(unanimous_errors),
        'majority_wrong': len(majority_errors),
        'all_correct': len(error_df[error_df['difficulty_score'] == 0]),
    })

print('\n\n=== ERROR ANALIZI OZETI ===')
display(pd.DataFrame(error_summaries))


ERROR ANALIZI: General
  Hold-out test satirlari: 632
  Tum modeller yanlis (score=5): 8 satir
  Cogunluk yanlis (score>=3): 26 satir

  Difficulty score dagilimi:
  {0: 576, 1: 18, 2: 12, 3: 12, 4: 6, 5: 8}

  --- TUM MODELLER YANLIS (8 satir) ---


,y_true,difficulty_score,base__chrom,base__pos,base__ref_base,base__alt_base,clinvar__sig
2403,0.0,5,chr14,95096544,C,T,Likely benign
39,1.0,5,chr1,11139429,C,A,Likely pathogenic
579,0.0,5,chr2,166044034,G,A,Likely benign
3831,0.0,5,chr20,44624217,A,T,Likely benign
1155,1.0,5,chr7,107701101,G,A,Likely pathogenic
3838,0.0,5,chr21,34792304,G,A,Likely benign
2746,0.0,5,chr17,7670706,G,A,Likely benign
2026,0.0,5,chr12,120989009,G,A,Likely benign



ERROR ANALIZI: Hereditary_Cancer
  Hold-out test satirlari: 143
  Tum modeller yanlis (score=5): 1 satir
  Cogunluk yanlis (score>=3): 4 satir

  Difficulty score dagilimi:
  {0: 120, 1: 13, 2: 6, 3: 1, 4: 2, 5: 1}

  --- TUM MODELLER YANLIS (1 satir) ---


,y_true,difficulty_score,base__chrom,base__pos,base__ref_base,base__alt_base,clinvar__sig
2767,0.0,5,chr17,7673836,C,T,Likely benign




=== ERROR ANALIZI OZETI ===


,panel,n_test,all_wrong,majority_wrong,all_correct
0,General,632,8,26,576
1,Hereditary_Cancer,143,1,4,120


In [13]:
# Cell 6: Gorsellestirme
fig_paths = []

# 1. Per-panel F1 karsilastirma
fig, ax = plt.subplots(figsize=(12, 6))
pivot = results_df.pivot(index='model', columns='panel', values='f1')
pivot.plot(kind='bar', ax=ax, colormap='Set2')
ax.set_title('Hold-Out F1 Score - Panel x Model (With In-Silico)', fontsize=14)
ax.set_ylabel('F1 Score')
ax.set_xlabel('Model')
ax.legend(title='Panel')
ax.set_ylim(0, 1.05)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8)
plt.tight_layout()
p = os.path.join(RESULTS_DIR, 'f1_comparison.png')
fig.savefig(p, dpi=150); fig_paths.append(p)
plt.show()

# 2. Feature Importance (LightGBM & XGBoost, her panel icin)
for panel in TRAIN_PANELS:
    for mt_name in ['lightgbm', 'xgboost']:
        key = (panel, mt_name)
        if key not in trained_models:
            continue
        model_obj = trained_models[key][0]

        if mt_name == 'lightgbm':
            importances = model_obj.feature_importances_
            feature_names = model_obj.feature_name_
        else:  # xgboost
            importances = model_obj.feature_importances_
            feature_names = model_obj.get_booster().feature_names
            if feature_names is None:
                feature_names = [f'f{i}' for i in range(len(importances))]

        # Top 20 features
        fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
        fi_df = fi_df.sort_values('importance', ascending=False).head(20)

        fig, ax = plt.subplots(figsize=(10, 8))
        ax.barh(range(len(fi_df)), fi_df['importance'].values, color='#4CAF50' if mt_name == 'lightgbm' else '#2196F3')
        ax.set_yticks(range(len(fi_df)))
        ax.set_yticklabels(fi_df['feature'].values, fontsize=9)
        ax.invert_yaxis()
        ax.set_title(f'Feature Importance - {panel} ({mt_name.upper()})', fontsize=13)
        ax.set_xlabel('Importance')
        plt.tight_layout()
        p = os.path.join(RESULTS_DIR, f'{panel}_{mt_name}_feature_importance.png')
        fig.savefig(p, dpi=150); fig_paths.append(p)
        plt.show()

# 3. Error heatmap per panel
for panel in TRAIN_PANELS:
    data = all_predictions[panel]
    y_true = data['y_true']
    preds = data['preds']

    heat_df = pd.DataFrame(index=y_true.index)
    for mt_name, pred_series in preds.items():
        heat_df[mt_name] = (pred_series.values != y_true.values).astype(int)
    heat_df['difficulty'] = heat_df.sum(axis=1)
    heat_df = heat_df.sort_values('difficulty', ascending=False)
    heat_df = heat_df.drop(columns=['difficulty'])

    if len(heat_df) > 100:
        heat_df = heat_df.head(100)

    fig, ax = plt.subplots(figsize=(8, max(6, len(heat_df) * 0.08)))
    sns.heatmap(heat_df, cmap=['#4CAF50', '#F44336'], cbar_kws={'label': '0=Dogru, 1=Yanlis'},
                ax=ax, yticklabels=False)
    ax.set_title(f'Error Heatmap - {panel} (en zor satirlar)', fontsize=12)
    ax.set_xlabel('Model')
    ax.set_ylabel('Test satirlari')
    plt.tight_layout()
    p = os.path.join(RESULTS_DIR, f'{panel}_error_heatmap.png')
    fig.savefig(p, dpi=150); fig_paths.append(p)
    plt.show()

# 4. Confusion matrix (en iyi model, her panel)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

for panel in TRAIN_PANELS:
    panel_res = results_df[results_df['panel'] == panel]
    best_row = panel_res.loc[panel_res['f1'].idxmax()]
    best_mt = best_row['model']

    y_true = all_predictions[panel]['y_true']
    y_pred = all_predictions[panel]['preds'][best_mt]

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Pathogenic']).plot(ax=ax, cmap='Blues')
    ax.set_title(f'Confusion Matrix - {panel} ({best_mt})', fontsize=12)
    plt.tight_layout()
    p = os.path.join(RESULTS_DIR, f'{panel}_confusion_matrix.png')
    fig.savefig(p, dpi=150); fig_paths.append(p)
    plt.show()

# 4. CFTR recall bar chart
if cftr_results:
    cftr_df = pd.DataFrame(cftr_results)
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(cftr_df['model'], cftr_df['recall'], color='#FF9800')
    ax.bar_label(bars, fmt='%.3f')
    ax.set_title('CFTR Recall (General Modeli ile)', fontsize=14)
    ax.set_ylabel('Recall')
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    p = os.path.join(RESULTS_DIR, 'cftr_recall.png')
    fig.savefig(p, dpi=150); fig_paths.append(p)
    plt.show()

# 6. Difficulty score dagilim histogrami
fig, axes = plt.subplots(1, len(TRAIN_PANELS), figsize=(5*len(TRAIN_PANELS), 5))
if len(TRAIN_PANELS) == 1:
    axes = [axes]
for ax, panel in zip(axes, TRAIN_PANELS):
    err_csv = pd.read_csv(os.path.join(RESULTS_DIR, f'{panel}_error_analysis.csv'), index_col=0)
    err_csv['difficulty_score'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#2196F3')
    ax.set_title(f'{panel} - Difficulty Score Dagilimi')
    ax.set_xlabel('Difficulty Score (0-5)')
    ax.set_ylabel('Satir Sayisi')
plt.tight_layout()
p = os.path.join(RESULTS_DIR, 'difficulty_distribution.png')
fig.savefig(p, dpi=150); fig_paths.append(p)
plt.show()

print(f'\nToplam {len(fig_paths)} grafik kaydedildi.')


Toplam 11 grafik kaydedildi.


In [14]:
# Cell 7: PDF Rapor Olusturma
from fpdf import FPDF
from datetime import datetime


class PanelReport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 10)
        self.cell(0, 8, 'Teknofest - Single Panel Training Report (With In-Silico)', align='C', new_x='LMARGIN', new_y='NEXT')
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 10, f'Sayfa {self.page_no()}/{{nb}}', align='C')


pdf = PanelReport()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=20)

# --- Baslik sayfasi ---
pdf.add_page()
pdf.set_font('Helvetica', 'B', 20)
pdf.ln(40)
pdf.cell(0, 15, 'Single-Panel Model Egitim Raporu', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 14)
pdf.cell(0, 10, 'In-Silico Skorlar ILE (v3)', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.ln(10)
pdf.set_font('Helvetica', '', 11)
pdf.cell(0, 8, f'Tarih: {datetime.now().strftime("%Y-%m-%d %H:%M")}', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 8, f'Modeller: LightGBM, XGBoost, NN, DNN, SVM', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 8, f'Egitim: {" / ".join(TRAIN_PANELS)} | Test-only: {" / ".join(TEST_ONLY_PANELS + ["CFTR"])}', align='C', new_x='LMARGIN', new_y='NEXT')

# --- Veri ozeti ---
pdf.add_page()
pdf.set_font('Helvetica', 'B', 14)
pdf.cell(0, 10, '1. Veri Ozeti', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 10)
pdf.cell(0, 7, f'Toplam veri: {len(df_raw)} satir', new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 7, f'Feature sayisi (FE sonrasi): {df_v3.shape[1] - 2} (target ve Panel haric)', new_x='LMARGIN', new_y='NEXT')
pdf.ln(3)
pdf.set_font('Helvetica', 'B', 11)
pdf.cell(0, 7, 'Panel Dagilimi:', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 10)
for p_name in TRAIN_PANELS + TEST_ONLY_PANELS + ['CFTR']:
    p_df = df_v3[df_v3['Panel'] == p_name]
    n_pos = int(p_df['target'].sum()) if 'target' in p_df.columns else 0
    n_neg = len(p_df) - n_pos
    pdf.cell(0, 6, f'  {p_name}: {len(p_df)} satir (pos={n_pos}, neg={n_neg})', new_x='LMARGIN', new_y='NEXT')

# --- Model sonuc tablosu ---
pdf.add_page()
pdf.set_font('Helvetica', 'B', 14)
pdf.cell(0, 10, '2. Hold-Out Sonuclari', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 9)

# Tablo basligi
col_widths = [30, 25, 20, 20, 20, 20, 20, 20, 15]
headers = ['Panel', 'Model', 'F1', 'AUC-ROC', 'AUC-PR', 'Prec', 'Recall', 'MCC', 'Sec']
pdf.set_font('Helvetica', 'B', 9)
for w, h in zip(col_widths, headers):
    pdf.cell(w, 7, h, border=1, align='C')
pdf.ln()

pdf.set_font('Helvetica', '', 8)
for _, row in results_df.iterrows():
    vals = [row['panel'], row['model'],
            f"{row['f1']:.4f}", f"{row['auc_roc']:.4f}", f"{row['auc_pr']:.4f}",
            f"{row['precision']:.4f}", f"{row['recall']:.4f}", f"{row['mcc']:.4f}",
            f"{row['elapsed_sec']:.0f}"]
    for w, v in zip(col_widths, vals):
        pdf.cell(w, 6, str(v), border=1, align='C')
    pdf.ln()

# En iyi model per panel
pdf.ln(5)
pdf.set_font('Helvetica', 'B', 11)
pdf.cell(0, 7, 'En Iyi Model (Panel Bazinda):', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 10)
for panel in TRAIN_PANELS:
    panel_res = results_df[results_df['panel'] == panel]
    best_row = panel_res.loc[panel_res['f1'].idxmax()]
    pdf.cell(0, 6, f'  {panel}: {best_row["model"]} (F1={best_row["f1"]:.4f})',
             new_x='LMARGIN', new_y='NEXT')

# --- Test-only panel sonuclari ---
if cftr_results:
    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 14)
    pdf.cell(0, 10, '3. Test-Only Panel Sonuclari (General modeli ile)', new_x='LMARGIN', new_y='NEXT')
    pdf.ln(3)

    col_w = [28, 22, 18, 18, 18, 18, 18, 18, 18, 12]
    hdrs = ['Panel', 'Model', 'F1', 'AUC-ROC', 'AUC-PR', 'Prec', 'Recall', 'MCC', 'Bal.Acc', 'N']
    pdf.set_font('Helvetica', 'B', 8)
    for w, h in zip(col_w, hdrs):
        pdf.cell(w, 7, h, border=1, align='C')
    pdf.ln()
    pdf.set_font('Helvetica', '', 7)
    for r in cftr_results:
        vals = [r['test_panel'], r['model'],
                f"{r['f1']:.4f}", f"{r.get('auc_roc',0):.4f}", f"{r.get('auc_pr',0):.4f}",
                f"{r['precision']:.4f}", f"{r['recall']:.4f}", f"{r.get('mcc',0):.4f}",
                f"{r.get('balanced_accuracy',0):.4f}", str(r['n_test'])]
        for w, v in zip(col_w, vals):
            pdf.cell(w, 6, v, border=1, align='C')
        pdf.ln()

    pdf.ln(3)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.cell(0, 7, 'En Iyi Model (Test-Only Panel Bazinda):', new_x='LMARGIN', new_y='NEXT')
    pdf.set_font('Helvetica', '', 9)
    cftr_df_pdf = pd.DataFrame(cftr_results)
    for tp in cftr_df_pdf['test_panel'].unique():
        tp_sub = cftr_df_pdf[cftr_df_pdf['test_panel'] == tp]
        best = tp_sub.loc[tp_sub['f1'].idxmax()]
        pdf.cell(0, 6, f'  {tp}: {best["model"]} (F1={best["f1"]:.4f}, Recall={best["recall"]:.4f})',
                 new_x='LMARGIN', new_y='NEXT')

# --- Error analizi ---
pdf.add_page()
pdf.set_font('Helvetica', 'B', 14)
pdf.cell(0, 10, '4. Cross-Model Error Analizi', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 10)

for es in error_summaries:
    pdf.cell(0, 7, f"{es['panel']}: {es['n_test']} test satiri | "
             f"Tum yanlis: {es['all_wrong']} | Cogunluk yanlis: {es['majority_wrong']} | "
             f"Tum dogru: {es['all_correct']}", new_x='LMARGIN', new_y='NEXT')

pdf.ln(5)
pdf.set_font('Helvetica', 'I', 9)
pdf.cell(0, 6, 'Not: Tum modellerin yanlis buldugu satirlar ground truth acisindan supheli olabilir.',
         new_x='LMARGIN', new_y='NEXT')

# --- Grafikler ---
for fp in fig_paths:
    if os.path.exists(fp):
        pdf.add_page()
        fname = os.path.basename(fp).replace('.png', '').replace('_', ' ').title()
        pdf.set_font('Helvetica', 'B', 12)
        pdf.cell(0, 10, fname, new_x='LMARGIN', new_y='NEXT')
        try:
            pdf.image(fp, x=10, w=190)
        except Exception as e:
            pdf.set_font('Helvetica', '', 10)
            pdf.cell(0, 7, f'Grafik yuklenemedi: {e}', new_x='LMARGIN', new_y='NEXT')

# Kaydet
report_path = os.path.join(REPORTS_DIR, 'v3_with_insil_panels_report.pdf')
pdf.output(report_path)
print(f'PDF rapor kaydedildi: {report_path}')

PDF rapor kaydedildi: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\reports\v3_with_insil_panels_report.pdf
